In [ ]:
"""
Assignment: Modern OpenGL & GLSL - Interactive 3D Rotatable Letter
File: HW3_RotateLetter.py (or save as .ipynb)

Dependencies:
    uv add glfw pyopengl numpy  (run with: uv run, or select the .venv kernel)
"""

import sys
import ctypes
import numpy as np
import glfw
from OpenGL.GL import *

# ==============================================================================
# GLOBAL STATE & INTERACTION VARIABLES
# ==============================================================================
WINDOW_WIDTH = 800
WINDOW_HEIGHT = 800

# Mouse tracking variables
is_dragging = False
last_mouse_x = 0.0
last_mouse_y = 0.0

# Current rotation angles around the axes (in degrees or radians)
angle_x = 0.0  # Controlled by vertical mouse drag (pitch)
angle_y = 0.0  # Controlled by horizontal mouse drag (yaw)


# ==============================================================================
# STEP 1: DEFINE LETTER GEOMETRY & INDICES
# ==============================================================================
def get_letter_geometry():
    """
    Returns:
        raw_vertices: np.ndarray (N, 2), float32 - Unscaled 2D (x, y) coordinates
        colors:       np.ndarray (N, 3), float32 - RGB values in [0.0, 1.0] per vertex
        indices:      np.ndarray (M, 3), uint32  - Triangle index triplets (minimum 7 triangles)
    """
    # -------------------------------------------------------------------------
    # TODO 1.1: Define your raw 2D vertex coordinates from your graph design.
    # Requirements:
    #   - Letter must consist of at least 7 triangles.
    #   - Ensure vertices do not form degenerate/collinear triangles.
    # -------------------------------------------------------------------------
    # Letter "N": labeled A-J going around the outline counter-clockwise.
    # C and H sit exactly on the stems' inner edges, so they are shared by
    # the stem and diagonal triangles (no T-junctions -> no color seams).
    raw_vertices = np.array([
        [  0.0,   0.0],  # 0 A - left stem, bottom-left
        [ 60.0,   0.0],  # 1 B - left stem, bottom-right
        [ 60.0, 210.0],  # 2 C - left stem inner edge, diagonal's lower start
        [240.0,   0.0],  # 3 D - right stem, bottom-left (diagonal's lower end)
        [300.0,   0.0],  # 4 E - right stem, bottom-right
        [300.0, 300.0],  # 5 F - right stem, top-right
        [240.0, 300.0],  # 6 G - right stem, top-left
        [240.0,  90.0],  # 7 H - right stem inner edge, diagonal's upper end
        [ 60.0, 300.0],  # 8 I - left stem, top-right (diagonal's upper start)
        [  0.0, 300.0],  # 9 J - left stem, top-left
    ], dtype=np.float32)

    # -------------------------------------------------------------------------
    # TODO 1.2: Assign a unique RGB color to each vertex for smooth interpolation.
    # Must have the same length (N) as raw_vertices.
    # -------------------------------------------------------------------------
    colors = np.array([
        [0.0, 0.0, 1.0],  # A - blue
        [1.0, 1.0, 0.0],  # B - yellow
        [1.0, 0.5, 0.0],  # C - orange
        [1.0, 0.0, 1.0],  # D - magenta
        [0.0, 1.0, 1.0],  # E - cyan
        [0.0, 1.0, 0.0],  # F - green
        [0.5, 0.0, 1.0],  # G - violet
        [1.0, 0.0, 0.5],  # H - pink
        [1.0, 0.0, 0.0],  # I - red
        [0.0, 0.6, 0.3],  # J - teal
    ], dtype=np.float32)

    # -------------------------------------------------------------------------
    # TODO 1.3: Define your triangle index connectivity array.
    # Each row contains 3 indices referencing rows in raw_vertices.
    # Must have AT LEAST 7 triangles (7 rows).
    # -------------------------------------------------------------------------
    # 8 triangles, all wound counter-clockwise.
    indices = np.array([
        # Left stem
        [0, 1, 2],  # A B C
        [0, 2, 9],  # A C J
        [2, 8, 9],  # C I J
        # Diagonal band
        [2, 3, 7],  # C D H
        [2, 7, 8],  # C H I
        # Right stem
        [3, 4, 7],  # D E H
        [7, 4, 5],  # H E F
        [7, 5, 6],  # H F G
    ], dtype=np.uint32)

    return raw_vertices, colors, indices


def validate_geometry(vertices, colors, indices):
    """Catches broken geometry before it reaches the GPU."""
    assert len(vertices) == len(colors), "Need exactly one color per vertex"
    assert len(indices) >= 7, "Letter must have at least 7 triangles"
    assert indices.max() < len(vertices), "Index references a missing vertex"
    assert len(np.unique(colors, axis=0)) == len(colors), "Vertex colors must be unique"
    for tri in indices:
        a, b, c = vertices[tri]
        area = 0.5 * ((b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0]))
        assert abs(area) > 1e-6, f"Degenerate (collinear) triangle: {tri}"


# ==============================================================================
# STEP 2: STANDARDIZE / FIT THE LETTER TO THE VIEWPORT
# ==============================================================================
def StandardizePoints(vertices, s=0.75):
    """
    Normalizes arbitrary 2D vertex coordinates into OpenGL's NDC range [-1, 1].

    Args:
        vertices: np.ndarray of shape (N, 2)
        s: float - target scale factor within [-1, 1] (e.g., 0.75 leaves a 25% margin)

    Returns:
        normalized_vertices: np.ndarray of shape (N, 2), centered at (0, 0)
    """
    # -------------------------------------------------------------------------
    # TODO 2: Implement the bounding-box normalization subroutine:
    #   1. Find min and max for both X and Y coordinates across all vertices.
    #   2. Compute the bounding-box center: center = (min + max) / 2.0.
    #   3. Translate vertices so the center is at (0, 0): centered = vertices - center.
    #   4. Compute the maximum span across dimensions: max_span = max(max_x - min_x, max_y - min_y).
    #   5. Scale uniformly by (2.0 * s) / max_span to fit within [-s, s] without distortion.
    # -------------------------------------------------------------------------
    
    min_xy = vertices.min(axis=0)
    max_xy = vertices.max(axis=0)
    center = (min_xy + max_xy) / 2.0
    max_span = np.max(max_xy - min_xy)  # one uniform scale -> no stretching
    normalized_vertices = (vertices - center) * (2.0 * s / max_span)

    return normalized_vertices.astype(np.float32)


# ==============================================================================
# STEP 3 & 6: GLSL SHADER SOURCES
# ==============================================================================
VERTEX_SHADER_SOURCE = """
#version 330 core

layout (location = 0) in vec2 aPos;
layout (location = 1) in vec3 aColor;

out vec3 vColor;

// -----------------------------------------------------------------------------
// TODO 6.1: Declare the uniform 4x4 transformation matrix ('view')
// -----------------------------------------------------------------------------
uniform mat4 view;

void main()
{
    vColor = aColor;
    
    // -------------------------------------------------------------------------
    // TODO 6.2: Transform the input position using the uniform matrix:
    // gl_Position = view * vec4(aPos, 0.0, 1.0);
    // -------------------------------------------------------------------------
    gl_Position = view * vec4(aPos, 0.0, 1.0);
}
"""

FRAGMENT_SHADER_SOURCE = """
#version 330 core

in vec3 vColor;
out vec4 FragColor;

void main()
{
    FragColor = vec4(vColor, 1.0);
}
"""


def compile_shader(source, shader_type):
    shader = glCreateShader(shader_type)
    glShaderSource(shader, source)
    glCompileShader(shader)
    if not glGetShaderiv(shader, GL_COMPILE_STATUS):
        error = glGetShaderInfoLog(shader).decode()
        raise RuntimeError(f"Shader compilation failed: {error}")
    return shader


def create_shader_program(vs_src, fs_src):
    vs = compile_shader(vs_src, GL_VERTEX_SHADER)
    fs = compile_shader(fs_src, GL_FRAGMENT_SHADER)
    
    program = glCreateProgram()
    glAttachShader(program, vs)
    glAttachShader(program, fs)
    glLinkProgram(program)
    
    if not glGetProgramiv(program, GL_LINK_STATUS):
        error = glGetProgramInfoLog(program).decode()
        raise RuntimeError(f"Program link failed: {error}")
        
    glDeleteShader(vs)
    glDeleteShader(fs)
    return program


# ==============================================================================
# STEP 5 & 6: MATRIX GENERATION & MOUSE INTERACTION
# ==============================================================================
def create_rotation_matrix(rot_x_deg, rot_y_deg):
    """
    Builds a 4x4 compound rotation matrix (R = Rx * Ry) using 3D Euler angles.

    Returns:
        np.ndarray of shape (4, 4), float32 in column-major order for OpenGL.
    """
    rad_x = np.radians(rot_x_deg)
    rad_y = np.radians(rot_y_deg)

    # -------------------------------------------------------------------------
    # TODO 6.3: Build the 4x4 rotation matrices Rx and Ry:
    #
    #   Rx (pitch around X-axis):
    #   [ 1,          0,           0,  0 ]
    #   [ 0,  cos(rad_x), -sin(rad_x),  0 ]
    #   [ 0,  sin(rad_x),  cos(rad_x),  0 ]
    #   [ 0,          0,           0,  1 ]
    #
    #   Ry (yaw around Y-axis):
    #   [  cos(rad_y), 0, sin(rad_y), 0 ]
    #   [           0, 1,          0, 0 ]
    #   [ -sin(rad_y), 0, cos(rad_y), 0 ]
    #   [           0, 0,          0, 1 ]
    #
    # Multiply them together: compound_rotation = Rx @ Ry (or Ry @ Rx).
    # -------------------------------------------------------------------------
    
    cx, sx = np.cos(rad_x), np.sin(rad_x)
    cy, sy = np.cos(rad_y), np.sin(rad_y)

    Rx = np.array([
        [1,  0,   0, 0],
        [0, cx, -sx, 0],
        [0, sx,  cx, 0],
        [0,  0,   0, 1],
    ], dtype=np.float32)

    Ry = np.array([
        [ cy, 0, sy, 0],
        [  0, 1,  0, 0],
        [-sy, 0, cy, 0],
        [  0, 0,  0, 1],
    ], dtype=np.float32)

    compound_rotation = Rx @ Ry

    # NumPy is row-major; glUniformMatrix4fv(..., GL_FALSE, ...) expects
    # column-major, so transpose before handing it to OpenGL.
    return np.ascontiguousarray(compound_rotation.T)


def mouse_button_callback(window, button, action, mods):
    """Captures mouse press and release events."""
    global is_dragging, last_mouse_x, last_mouse_y
    if button == glfw.MOUSE_BUTTON_LEFT:
        if action == glfw.PRESS:
            is_dragging = True
            last_mouse_x, last_mouse_y = glfw.get_cursor_pos(window)
        elif action == glfw.RELEASE:
            is_dragging = False


def cursor_position_callback(window, xpos, ypos):
    """Captures mouse movements while dragging."""
    global is_dragging, last_mouse_x, last_mouse_y, angle_x, angle_y
    if not is_dragging:
        return

    # -------------------------------------------------------------------------
    # TODO 5: Calculate cursor deltas and map them to rotation angles:
    #   - dx = xpos - last_mouse_x
    #   - dy = ypos - last_mouse_y
    #   - Left/Right drag (dx) updates angle_y (rotation around Y-axis).
    #   - Up/Down drag (dy) updates angle_x (rotation around X-axis).
    #   - Use a sensitivity factor (e.g., sensitivity = 0.5) to keep motion smooth.
    #   - Update last_mouse_x and last_mouse_y to the current position.
    # -------------------------------------------------------------------------
    dx = xpos - last_mouse_x
    dy = ypos - last_mouse_y
    
    sensitivity = 0.5
    angle_y += dx * sensitivity
    angle_x += dy * sensitivity

    last_mouse_x = xpos
    last_mouse_y = ypos


# ==============================================================================
# MAIN PIPELINE EXECUTION
# ==============================================================================
def main():
    if not glfw.init():
        print("Failed to initialize GLFW", file=sys.stderr)
        return

    # Request OpenGL 3.3 Core Profile
    glfw.window_hint(glfw.CONTEXT_VERSION_MAJOR, 3)
    glfw.window_hint(glfw.CONTEXT_VERSION_MINOR, 3)
    glfw.window_hint(glfw.OPENGL_PROFILE, glfw.OPENGL_CORE_PROFILE)
    glfw.window_hint(glfw.OPENGL_FORWARD_COMPAT, GL_TRUE)  # Required for macOS

    window = glfw.create_window(
        WINDOW_WIDTH, WINDOW_HEIGHT,
        "HW3: Rotate Letter - Noah Crouch",
        None, None
    )
    if not window:
        glfw.terminate()
        raise RuntimeError("Failed to create GLFW window")

    glfw.make_context_current(window)

    # Register mouse callbacks
    glfw.set_mouse_button_callback(window, mouse_button_callback)
    glfw.set_cursor_pos_callback(window, cursor_position_callback)

    # Enable Depth Testing for correct 3D occlusions during rotation
    glEnable(GL_DEPTH_TEST)
    glDepthFunc(GL_LESS)

    # Compile Shaders
    shader_program = create_shader_program(VERTEX_SHADER_SOURCE, FRAGMENT_SHADER_SOURCE)
    view_loc = glGetUniformLocation(shader_program, "view")

    # Fetch geometry and normalize points
    raw_verts, colors, indices = get_letter_geometry()
    validate_geometry(raw_verts, colors, indices)
    # s=0.7 keeps rotated corners (radius s*sqrt(2) ~= 0.99) inside the z clip range
    norm_verts = StandardizePoints(raw_verts, s=0.7)

    # Interleave vertex data: [x, y, r, g, b] per vertex (5 floats per vertex)
    interleaved_data = np.hstack([norm_verts, colors]).astype(np.float32).flatten()
    index_data = indices.flatten().astype(np.uint32)
    index_count = len(index_data)

    # -------------------------------------------------------------------------
    # TODO 3: Set up VAO, VBO, and EBO buffers on the GPU:
    #   1. Generate and bind a Vertex Array Object (VAO): glGenVertexArrays, glBindVertexArray.
    #   2. Generate and bind a VBO (GL_ARRAY_BUFFER): glGenBuffers, glBindBuffer.
    #   3. Upload interleaved vertex data using glBufferData.
    #   4. Generate and bind an EBO (GL_ELEMENT_ARRAY_BUFFER): glGenBuffers, glBindBuffer.
    #   5. Upload index data to the EBO using glBufferData.
    #   6. Configure glVertexAttribPointer and glEnableVertexAttribArray:
    #      - Location 0 (Position): 2 floats (x, y), stride = 5 * sizeof(float), offset = 0
    #      - Location 1 (Color):    3 floats (r, g, b), stride = 5 * sizeof(float), offset = 2 * sizeof(float)
    # -------------------------------------------------------------------------
    vao = glGenVertexArrays(1)
    glBindVertexArray(vao)

    vbo = glGenBuffers(1)
    glBindBuffer(GL_ARRAY_BUFFER, vbo)
    glBufferData(GL_ARRAY_BUFFER, interleaved_data.nbytes, interleaved_data, GL_STATIC_DRAW)

    ebo = glGenBuffers(1)
    glBindBuffer(GL_ELEMENT_ARRAY_BUFFER, ebo)
    glBufferData(GL_ELEMENT_ARRAY_BUFFER, index_data.nbytes, index_data, GL_STATIC_DRAW)

    float_size = ctypes.sizeof(ctypes.c_float)
    stride = 5 * float_size

    # Position attribute (layout location 0)
    glVertexAttribPointer(0, 2, GL_FLOAT, GL_FALSE, stride, ctypes.c_void_p(0))
    glEnableVertexAttribArray(0)

    # Color attribute (layout location 1)
    glVertexAttribPointer(1, 3, GL_FLOAT, GL_FALSE, stride, ctypes.c_void_p(2 * float_size))
    glEnableVertexAttribArray(1)

    glBindVertexArray(0)  # Unbind VAO

    # -------------------------------------------------------------------------
    # RENDER LOOP
    # -------------------------------------------------------------------------
    while not glfw.window_should_close(window):
        # Clear screen and depth buffers
        glClearColor(0.12, 0.12, 0.14, 1.0)
        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

        glUseProgram(shader_program)

        # Compute transformation matrix based on mouse offsets
        rotation_matrix = create_rotation_matrix(angle_x, angle_y)

        # ---------------------------------------------------------------------
        # TODO 6.4: Upload rotation_matrix to uniform 'view' in shader:
        #   glUniformMatrix4fv(view_loc, 1, GL_FALSE, rotation_matrix)
        # ---------------------------------------------------------------------
        if view_loc != -1:
            glUniformMatrix4fv(view_loc, 1, GL_FALSE, rotation_matrix)

        # ---------------------------------------------------------------------
        # TODO 3.2: Draw elements using the bound EBO:
        #   1. Bind your VAO: glBindVertexArray(vao)
        #   2. Execute glDrawElements(GL_TRIANGLES, index_count, GL_UNSIGNED_INT, None)
        #   3. Unbind VAO
        # ---------------------------------------------------------------------
        glBindVertexArray(vao)
        glDrawElements(GL_TRIANGLES, index_count, GL_UNSIGNED_INT, None)
        glBindVertexArray(0)

        glfw.swap_buffers(window)
        glfw.poll_events()

    # Cleanup resources
    glDeleteVertexArrays(1, [vao])
    glDeleteBuffers(1, [vbo])
    glDeleteBuffers(1, [ebo])
    glDeleteProgram(shader_program)

    glfw.terminate()


if __name__ == "__main__":
    main()

: 